# Market analysis — daily read
**One notebook for all towns.** Set `POPULATION` below to `"sant_cugat"`, `"sant_quirze"` or `"cerdanyola"` and run all.

What it answers every day:
1. Is inventory in our band rising or falling? Are sellers cutting?
2. Where are asking prices vs **real closing prices** (notary data)?
3. Is this a good month to buy, or is waiting costing us money?
4. Which stale listings give us the most negotiating leverage right now?

All scraped prices are **asking** prices. Notary baselines are **closing** prices (May 2025 – Apr 2026 reports in `PENotariado_reports/`).

In [ ]:
POPULATION = "sant_cugat"   # "sant_cugat" | "sant_quirze" | "cerdanyola"

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display, Markdown

import analysis as an

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

cfg = an.POPULATIONS[POPULATION]
data = an.load_data(POPULATION)
feats = an.build_features(data)
hed = an.fit_hedonic(feats)
feats = hed["features"]
km = an.kaplan_meier(feats)
panel = an.build_daily_panel(data["properties"], data["history"], data["events"])
market = an.compute_market_daily(panel, feats, band=an.SEARCH_BAND)
baselines = an.load_notariado()
verdict = an.market_timing_verdict(market, baselines, POPULATION)
gap = an.gap_analysis(feats, baselines, POPULATION)

print(f"{cfg['label']} — {len(data['properties'])} tracked, "
      f"{(feats.status=='active').sum()} active rows, "
      f"{feats[feats.status=='active']['cluster_id'].nunique()} unique physical properties")
print(f"Hedonic model R² = {hed['r2']:.2f} on {hed['n_train']} listings")

## 1 · Snapshot (search band 400–700k)

In [ ]:
last = market.dropna(subset=["inventory"]).iloc[-1]
last28 = market.tail(28)
active_band = feats[(feats.status == "active") & feats.price.between(*an.SEARCH_BAND) & (feats.is_cluster_canonical == 1)]
target_band = active_band[active_band.price.between(*an.TARGET_BAND)]

display(Markdown(f'''
| Metric | Value |
|---|---|
| Active physical properties in band (400–700k) | **{len(active_band)}** ({len(target_band)} in 500–600k) |
| Median asking €/m² in band (7d) | **{last["median_ppsqm_7d"]:,.0f} €/m²** |
| New listings, last 28 days | {last28["new_listings"].sum():.0f} |
| Confirmed delistings, last 28 days | {last28["delistings"].sum():.0f} |
| Months of supply | **{last["months_of_supply"]}** (>6 buyer market, <4 seller market) |
| Share of stock with ≥1 price cut | **{last["cut_breadth"]:.0%}** |
| Multi-agency listed (leverage signal) | {int(active_band["multi_listed"].sum())} listings |
| Detected re-listings (fake "new") | {int(active_band["relist_count"].sum())} listings |
'''))

## 2 · Inventory & flows — is stock piling up?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
m = market.set_index("day")
axes[0].plot(m.index, m["inventory"], lw=2, color="#1f77b4")
axes[0].set_ylabel("Active listings (band)")
axes[0].set_title(f"{cfg['label']} — inventory in 400–700k band")
axes[0].grid(alpha=.3)
axes[1].bar(m.index, m["new_listings"], width=1, alpha=.6, label="New", color="#2ca02c")
axes[1].bar(m.index, -m["delistings"], width=1, alpha=.6, label="Delisted (confirmed)", color="#d62728")
axes[1].axhline(0, color="k", lw=.5)
axes[1].set_ylabel("Daily flow")
axes[1].legend(); axes[1].grid(alpha=.3)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

## 3 · Asking €/m² vs real closing €/m²
The spread between the blue line (what sellers ask) and the dashed lines (what buyers actually paid at the notary) is the **total room between listing fantasy and reality** — part composition, part negotiation margin.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(m.index, m["median_ppsqm"], alpha=.3, lw=1, color="#1f77b4")
ax.plot(m.index, m["median_ppsqm_7d"], lw=2.5, color="#1f77b4", label="Median ASKING €/m² (7d, our panel)")
if gap is not None:
    colors = {"sant_cugat": "#d62728", "zip_08173": "#ff7f0e", "zip_08195": "#9467bd"}
    for _, row in gap.iterrows():
        ax.axhline(row["closing_mean_eur_m2_12m"], ls="--", lw=1.5,
                   color=colors.get(row["zone"], "gray"),
                   label=f"CLOSING mean 12m — {row['zone']} ({row['closing_mean_eur_m2_12m']:,} €/m²)")
ax.set_ylabel("€/m²"); ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=.3)
ax.set_title("Asking (scraped) vs closing (notary) prices")
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

if gap is not None:
    display(gap)
    display(Markdown("**Reading:** the asking premium is an *upper bound* on negotiating margin — "
                     "closing data includes cheaper stock than what's currently listed. But a listing "
                     "priced ≥25% above zone closing average with weeks on market is objectively overpriced."))
else:
    print("No notary baselines for this population (download PENotariado reports to add them).")

## 4 · Seller capitulation — cuts & supply pressure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(m.index, m["cut_breadth"] * 100, lw=2, color="#d62728")
axes[0].set_title("Share of active stock with ≥1 price cut (%)")
axes[0].grid(alpha=.3)
axes[1].plot(m.index, m["months_of_supply"], lw=2, color="#8c564b")
axes[1].axhspan(0, 4, alpha=.08, color="red")
axes[1].axhspan(6, 12, alpha=.08, color="green")
axes[1].set_title("Months of supply (green zone = buyer market)")
axes[1].grid(alpha=.3)
for ax in axes: ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 5 · How fast does this market sell?
Kaplan–Meier survival of listings (excluding the ones that pre-date our scraping). If a listing has outlived most of its cohort, the market has rejected its price — that's your leverage.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.step(km["t"], km["survival"] * 100, where="post", lw=2)
for q, lbl in [(.75, "25% sold"), (.5, "half sold"), (.25, "75% sold")]:
    hit = km[km["survival"] <= q]
    if len(hit):
        d = hit["t"].iloc[0]
        ax.axvline(d, ls=":", color="gray", lw=1)
        ax.annotate(f"{lbl}\n{d:.0f}d", (d, q * 100), fontsize=9, ha="left")
ax.set_xlabel("Days listed"); ax.set_ylabel("% still on market")
ax.set_title(f"{cfg['label']} — listing survival"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6 · Verdict — is this a good month to buy?

In [ ]:
import json as _json
print(f"BUYER SCORE: {verdict['buyer_score']}   →  {verdict['verdict']}\n")
for name, sig in verdict["signals"].items():
    flag = "🟢" if sig.get("buyer_friendly") else "🔴"
    detail = {k: v for k, v in sig.items() if k != "buyer_friendly"}
    print(f"{flag} {name}: {_json.dumps(detail, ensure_ascii=False, default=str)}")

if POPULATION == "sant_cugat":
    muni = baselines["sant_cugat"]
    display(Markdown(f'''
### Where are we in the cycle? (notary closing data, 12 years)
- Median closing price went **2,045 €/m² (2014) → 4,925 €/m² (2025)**; the only dip in 12 years was **2023 (−1.7%)**, instantly recovered (+11% in 2024, +6.8% in 2025).
- Last 12 months: **4,569 €/m² mean**, 985 transactions, average deal **603k €** — your budget buys the *average* Sant Cugat home, not a premium one.
- Transactions are healthy (~1,000/yr): no demand collapse in sight; buyers need ~**7 years of net income**, the highest ever recorded here.
- **Implication:** waiting for a broad price drop has been a losing trade for a decade; the realistic play is not *timing the market* but *finding the mispriced/stale listing* — which is exactly what the leverage list below does.
'''))

## 7 · Your budget reality check

In [ ]:
profile = an.FinancialProfile()   # 6.5k income, 180k savings, 2%, 30y
rows = []
for price in [450_000, 500_000, 550_000, 575_000, 600_000, 620_000]:
    rows.append(an.affordability(price, profile))
display(pd.DataFrame(rows))
cap = an.max_affordable_price(profile)
cap_c = an.max_affordable_price(profile, payment=profile.comfortable_payment)
cap_s = an.max_affordable_price(profile, payment=profile.stretch_payment)
display(Markdown(f'''
- At **1,500 €/month** (comfortable): max price ≈ **{cap_c['binding_max_price']:,} €**
- At **1,700 €/month** (max): max price ≈ **{cap['binding_max_price']:,} €**
- At **1,800 €/month** (stretch): max price ≈ **{cap_s['binding_max_price']:,} €** — note the binding constraint above ~570k is **cash for the 20% down payment + 11.5% costs**, not the payment. Every +10k of savings raises your ceiling ~+29k.
- A 620k purchase needs ≈ **{an.affordability(620_000, profile)['cash_needed_at_80ltv']:,} € cash** at 80% LTV — plan for it only if the price started ≥680k and you negotiated hard.
'''))

## 8 · Leverage list — stale stock to attack now

In [ ]:
active = feats[(feats.status == "active") & feats.price.between(*an.SEARCH_BAND) & (feats.is_cluster_canonical == 1)].copy()
active["staleness_pctile"] = active["days_online_effective"].map(lambda d: round(an.staleness_percentile(km, d), 2))
leverage = an.add_offer_columns(active, km, verdict, gap)
cols = ["title", "price", "sqm", "rooms", "days_online_effective", "staleness_pctile",
        "n_cuts", "cum_discount_pct", "multi_listed", "relist_count", "kw_any_motivated",
        "hedonic_residual_pct", "est_margin_pct", "offer_opening", "offer_target", "bucket", "url"]
cols = [c for c in cols if c in leverage.columns]
display(leverage.sort_values("est_margin_pct", ascending=False)[cols].head(25))

---
### Caveats
- `days_online_effective` already undoes detected re-listings, but anything listed before scraping started (May/Jun 2026 cohort) is older than shown (`preexisting=1`).
- Delisting ≠ sold: withdrawals and expiries are included, so months-of-supply is optimistic (real absorption is slower).
- Notary data lags ~2–3 months and mixes flats and houses; use zone lines as anchors, not gospel.
- Re-run `enrich_details.py <population>` weekly so keyword flags (`kw_*`) have full descriptions to work with.